In [1]:
import re
import pandas as pd
import emoji
import contractions

from bs4 import BeautifulSoup

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [2]:
df = pd.read_excel("C://Users//koh18//Downloads//TNL Project//manual_labelled_clean_english.xlsx")
print(df.shape)
df.head()

(1139, 7)


,domain,topic,video_title,video_url,comment,sentiment,comment_en
0,AI,AI Hub,Malaysia as a generative AI hub - Interview wi...,https://www.youtube.com/watch?v=LZSUz-SuYgI,He knows what he is doing 👍👍 the good truth an...,Positive,He knows what he's doing 👍👍 the truth and the ...
1,AI,AI Hub,Malaysia as a generative AI hub - Interview wi...,https://www.youtube.com/watch?v=LZSUz-SuYgI,Industrial Master Plan\nShould be Target Certa...,Neutral,The industrial master plan should be targeted ...
2,AI,AI Hub,Malaysia as a generative AI hub - Interview wi...,https://www.youtube.com/watch?v=LZSUz-SuYgI,Very intelligent minister,Positive,Very intelligent minister
3,AI,AI Hub,Malaysia as a generative AI hub - Interview wi...,https://www.youtube.com/watch?v=LZSUz-SuYgI,The beginning of Malaysia AI Artificial Intell...,Positive,The beginning of Malaysia's AI (Artificial Int...
4,AI,AI Hub,Malaysia as a generative AI hub - Interview wi...,https://www.youtube.com/watch?v=LZSUz-SuYgI,Lagi MANTAP MALAYSIA..,Positive,Malaysia is really awesome again..


In [3]:
def normalize_text(text):
    text = BeautifulSoup(text, "html.parser").get_text()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    return text

def normalize_social_text(text):
    # Reduce repeated punctuation
    text = re.sub(r'([!?.,])\1{2,}', r'\1', text)
    # Reduce repeated letters
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1\1', text)
    return text

stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    if pd.isna(text):
        return ""
    text = str(text)
    # Remove HTML, URLs, Emails
    text = normalize_text(text)
    # Normalize social media writing
    text = normalize_social_text(text)
    # Expand contractions
    text = contractions.fix(text)
    # Convert emojis into words
    text = emoji.demojize(text, delimiters=(" ", " "))
    # Lowercase
    text = text.lower()
    # Remove invisible Unicode characters
    text = re.sub(r'[\u200B-\u200D\uFEFF]', '', text)
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    # Tokenize
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [
        word for word in tokens
        if word not in stop_words
    ]
    # Lemmatize
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
    ]
    return " ".join(tokens)

In [4]:
df["clean_comment"] = df["comment_en"].apply(preprocess)
df[["comment_en", "clean_comment"]].head(10)

,comment_en,clean_comment
0,He knows what he's doing 👍👍 the truth and the ...,know thumb thumb truth stronghold
1,The industrial master plan should be targeted ...,industrial master plan targeted certain area e...
2,Very intelligent minister,intelligent minister
3,The beginning of Malaysia's AI (Artificial Int...,beginning malaysia ai artificial intelligence ...
4,Malaysia is really awesome again..,malaysia really awesome
5,Congratulations on your ratification of the WT...,congratulation ratification wto law
6,Awesome,awesome
7,"We need results, not headlines.",need result headline
8,There are bigger issues to solve first.,bigger issue solve first
9,I don't think this is enough.,think enough


In [5]:
df.to_csv(
    "cleaned_training_dataset_normal.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Cleaned dataset saved successfully!")

Cleaned dataset saved successfully!
